In [2]:
"""
================================================================================
AGENTE INTELIGENTE COM LANGGRAPH - NOTEBOOK DIDÁTICO COMPLETO
================================================================================

📚 CONTEÚDO DESTE NOTEBOOK:
1. Arquitetura geral do sistema
2. Instalação e imports explicados
3. Criação de ferramentas (Tools)
4. Configuração do modelo LLM
5. Definição do estado do grafo
6. Funções dos nós (cérebro do grafo)
7. Construção do grafo
8. Função de execução
9. Exemplos práticos
10. Exercícios para praticar

🎯 OBJETIVO:
Criar um agente que responde perguntas, usa ferramentas quando necessário,
e retorna respostas em formato estruturado (contexto + explicação + resposta).

================================================================================
"""

# ============================================================================
# 📦 SEÇÃO 1: ARQUITETURA GERAL
# ============================================================================
"""
FLUXO DO SISTEMA:

    ┌─────────────┐
    │   Usuário   │ "Calcule 10+5 e traduza 'olá'"
    └──────┬──────┘
           │
           ▼
    ┌─────────────────────────────────────────┐
    │         LANGGRAPH (Grafo de Estados)     │
    │                                          │
    │  ┌──────┐    ┌───────────┐    ┌────────┐│
    │  │Agente│───▶│Ferramentas│───▶│Formatar││
    │  └──────┘    └───────────┘    └────────┘│
    │     ▲             │                      │
    │     └─────────────┘                      │
    │    (pode repetir várias vezes)          │
    └─────────────────────────────────────────┘
           │
           ▼
       📄 Resposta Formatada em 3 Parágrafos

PADRÃO REACT (Reasoning + Acting):
1. REASON (Raciocinar): "Preciso calcular e traduzir"
2. ACT (Agir): Chama ferramentas necessárias
3. OBSERVE (Observar): Recebe resultados das ferramentas
4. REASON (Raciocinar): "Agora tenho tudo para responder"
5. ACT (Agir): Formata resposta final
"""

# ============================================================================
# 📦 SEÇÃO 2: INSTALAÇÃO E IMPORTS
# ============================================================================
"""
INSTALAÇÃO (execute no terminal ou célula anterior):
!pip install langgraph langchain langchain-openai python-dotenv
"""

# ---------- IMPORTS BÁSICOS ----------
import os  
# ↑ Para gerenciar variáveis de ambiente (API keys, configurações)

import operator  
# ↑ Para operações em listas/dicts (usado internamente pelo LangGraph)

import re  
# ↑ Regex para validação de formato de texto

# ---------- IMPORTS DE TIPAGEM ----------
from typing import TypedDict, Annotated, Sequence
# ↑ TypedDict: Define estruturas de dados com tipos específicos
# ↑ Annotated: Adiciona metadados aos tipos (ex: como combinar listas)
# ↑ Sequence: Tipo genérico para sequências (listas, tuplas, etc)

# ---------- IMPORTS DO LANGCHAIN CORE ----------
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage
"""
CLASSES DE MENSAGENS:
- BaseMessage: Classe base para todas as mensagens
- HumanMessage: Mensagem do usuário/humano
  Exemplo: HumanMessage(content="Calcule 10+5")
  
- AIMessage: Resposta do modelo de IA
  Exemplo: AIMessage(content="O resultado é 15")
  Pode conter tool_calls quando o LLM decide usar ferramentas
  
- ToolMessage: Resultado da execução de uma ferramenta
  Exemplo: ToolMessage(content="Resultado: 15", name="calculadora")
"""

from langchain_openai import ChatOpenAI
# ↑ Cliente para conectar aos modelos da OpenAI (GPT-4, GPT-3.5, etc)
# Exemplo de uso: ChatOpenAI(model="gpt-4o-mini", temperature=0)

from langchain_core.tools import tool
# ↑ Decorator @tool que transforma funções Python em ferramentas
# O LLM consegue "ver" essas ferramentas e decidir quando usá-las

# ---------- IMPORTS DO LANGGRAPH ----------
from langgraph.graph import StateGraph, END
"""
COMPONENTES DO GRAFO:
- StateGraph: Cria um grafo onde cada nó processa e atualiza um estado
  É como um fluxograma inteligente onde dados fluem entre nós
  
- END: Constante especial que marca o fim da execução do grafo
  Quando um nó aponta para END, o grafo termina
"""

from langgraph.prebuilt import ToolNode
"""
ToolNode: Nó pré-construído especializado em executar ferramentas
- Recebe tool_calls do LLM
- Executa as ferramentas correspondentes
- Retorna os resultados
Você não precisa implementar a lógica, apenas fornecer as ferramentas!
"""

from langgraph.graph.message import add_messages
"""
add_messages: Função reducer para combinar mensagens
Quando múltiplos nós retornam mensagens, add_messages sabe como combiná-las
de forma inteligente (adiciona novas mensagens ao histórico existente)
"""

# ============================================================================
# 🔑 SEÇÃO 3: CONFIGURAÇÃO DA API KEY
# ============================================================================

# IMPORTANTE: Substitua "sua-chave-aqui" pela sua chave real da OpenAI
os.environ["OPENAI_API_KEY"] = "sua-chave-aqui"

"""
ALTERNATIVAS PARA CONFIGURAR A API KEY:

Opção 1: Diretamente no código (como acima)
os.environ["OPENAI_API_KEY"] = "sk-..."

Opção 2: Arquivo .env (mais seguro)
1. Crie um arquivo .env na mesma pasta
2. Adicione: OPENAI_API_KEY=sk-...
3. Use: from dotenv import load_dotenv
         load_dotenv()

Opção 3: Variável de ambiente do sistema
export OPENAI_API_KEY=sk-...  (Linux/Mac)
set OPENAI_API_KEY=sk-...     (Windows)
"""

# ============================================================================
# 🛠️ SEÇÃO 4: CRIAÇÃO DAS FERRAMENTAS (TOOLS)
# ============================================================================

@tool
def calculadora(expressao: str) -> str:
    """
    Ferramenta para realizar cálculos matemáticos.
    
    ⚠️ A DOCSTRING É CRUCIAL! O LLM lê ela para decidir quando usar a tool.
    
    Args:
        expressao: Expressão matemática como string (ex: "(2+3)*4/2")
    
    Returns:
        Resultado do cálculo como string
    
    COMO FUNCIONA:
    1. Usuário pergunta: "Quanto é 10 + 5?"
    2. LLM lê esta docstring e identifica que pode calcular
    3. LLM chama: calculadora("10 + 5")
    4. Função executa e retorna: "Resultado: 15"
    5. LLM incorpora na resposta: "O resultado de 10 + 5 é 15"
    """
    try:
        # eval() avalia a string como código Python
        # SEGURANÇA: {"__builtins__": {}} remove funções perigosas
        # Isso previne código malicioso como: eval("__import__('os').system('rm -rf /')")
        resultado = eval(expressao, {"__builtins__": {}}, {})
        return f"Resultado: {resultado}"
    
    except Exception as e:
        # Captura erros (ex: divisão por zero, sintaxe inválida)
        return f"Erro no cálculo: {str(e)}"

"""
EXEMPLO DE USO DA CALCULADORA:

Entrada: calculadora("(100 + 50) / 3")
Saída: "Resultado: 50.0"

Entrada: calculadora("10 / 0")
Saída: "Erro no cálculo: division by zero"

LIMITAÇÕES:
- Não suporta funções complexas (sin, cos, sqrt) sem importar math
- Para expandir, você pode fazer:
  import math
  resultado = eval(expressao, {"__builtins__": {}, "math": math}, {})
"""


@tool
def tradutor(texto: str, idioma_destino: str = "inglês") -> str:
    """
    Ferramenta para traduzir textos entre português e inglês.
    
    ⚠️ NOTA DIDÁTICA: Esta é uma versão LIMITADA para estudo.
    Em produção, use APIs reais: Google Translate, DeepL, etc.
    
    Args:
        texto: Texto a ser traduzido
        idioma_destino: Idioma de destino ("inglês" ou "português")
    
    Returns:
        Texto traduzido
    
    POR QUE CRIAR ESTA TOOL SE O LLM JÁ SABE TRADUZIR?
    
    Resposta: Para fins DIDÁTICOS! Em produção, tools de tradução fazem sentido para:
    1. ✅ Usar API profissional (DeepL, Google) para consistência
    2. ✅ Implementar glossário corporativo específico
    3. ✅ Registrar traduções para auditoria/compliance
    4. ✅ Cache de traduções frequentes (economia de custos)
    
    Para seu caso de estudo, o LLM nativo traduz melhor que este dicionário!
    """
    
    # Dicionário Python simples (estrutura chave-valor)
    traducoes_pt_en = {
        "bom dia": "good morning",
        "boa tarde": "good afternoon",
        "boa noite": "good night",
        "olá": "hello",
        "tchau": "bye",
        "obrigado": "thank you",
        "por favor": "please",
    }
    
    # Dict comprehension: inverte o dicionário
    # {valor: chave for chave, valor in dict.items()}
    traducoes_en_pt = {v: k for k, v in traducoes_pt_en.items()}
    
    # Normalização: minúsculas + remove espaços extras
    texto_lower = texto.lower().strip()
    
    # Lógica de tradução
    if idioma_destino.lower() == "inglês":
        # .get(chave, valor_padrão) é mais seguro que dict[chave]
        # Se não achar, retorna o valor_padrão ao invés de dar erro
        return traducoes_pt_en.get(
            texto_lower, 
            f"[Tradução via LLM]: {texto} → (usar API de tradução real)"
        )
    
    elif idioma_destino.lower() == "português":
        return traducoes_en_pt.get(
            texto_lower, 
            f"[Tradução via LLM]: {texto} → (usar API de tradução real)"
        )
    
    else:
        return "Idioma não suportado. Use 'inglês' ou 'português'."

"""
EXEMPLO DE USO DO TRADUTOR:

Entrada: tradutor("bom dia", "inglês")
Saída: "good morning" ✅

Entrada: tradutor("cachorro", "inglês")
Saída: "[Tradução via LLM]: cachorro → (usar API de tradução real)" 
(não está no dicionário)

COMO EXPANDIR PARA PRODUÇÃO:

@tool
def tradutor_deepl(texto: str, idioma_destino: str) -> str:
    import deepl
    translator = deepl.Translator("SUA_API_KEY")
    result = translator.translate_text(texto, target_lang=idioma_destino)
    return result.text
"""

# Lista de todas as ferramentas disponíveis
tools = [calculadora, tradutor]

"""
COMO ADICIONAR MAIS FERRAMENTAS:

@tool
def buscar_clima(cidade: str) -> str:
    '''Busca a previsão do tempo para uma cidade'''
    # Implementação usando API de clima
    return f"Clima em {cidade}: 25°C, ensolarado"

@tool
def gerar_senha(tamanho: int = 12) -> str:
    '''Gera uma senha aleatória segura'''
    import random, string
    chars = string.ascii_letters + string.digits + string.punctuation
    return ''.join(random.choice(chars) for _ in range(tamanho))

# Adicione à lista
tools = [calculadora, tradutor, buscar_clima, gerar_senha]

# Pronto! O LLM agora pode usar todas essas ferramentas!
"""

# ============================================================================
# 🧠 SEÇÃO 5: CONFIGURAÇÃO DO MODELO LLM
# ============================================================================

# Inicializa o modelo de linguagem
llm = ChatOpenAI(
    model="gpt-4o-mini",  # Modelo a ser usado (pode ser gpt-4, gpt-3.5-turbo, etc)
    temperature=0         # Controla criatividade: 0=determinístico, 1=criativo
)

"""
PARÂMETROS DO ChatOpenAI:

- model: Qual modelo usar
  "gpt-4o-mini"     → Mais rápido e barato, boa qualidade
  "gpt-4"           → Mais inteligente, mais caro
  "gpt-3.5-turbo"   → Mais barato, menos capaz

- temperature: Controla aleatoriedade/criatividade
  0.0  → Sempre a mesma resposta (determinístico) - bom para tarefas precisas
  0.5  → Equilíbrio entre criatividade e consistência
  1.0+ → Muito criativo, respostas variadas - bom para escrita criativa

- max_tokens: Limite de tokens na resposta (padrão: infinito)
- top_p: Controla diversidade de tokens (alternativa à temperature)
"""

# Vincula as ferramentas ao modelo
llm_with_tools = llm.bind_tools(tools)

"""
O QUE É bind_tools()? 🎯

Quando você executa llm.bind_tools(tools), o sistema:

1. LÊ AS DOCSTRINGS de cada ferramenta
2. CRIA UM SCHEMA JSON descrevendo:
   - Nome da ferramenta
   - Descrição (da docstring)
   - Parâmetros e seus tipos
   - Quando usar a ferramenta

3. ENVIA ESSE SCHEMA ao LLM em CADA chamada

4. O LLM PODE DECIDIR chamar as ferramentas quando apropriado

EXEMPLO DO SCHEMA QUE O LLM "VÊ":

{
  "name": "calculadora",
  "description": "Ferramenta para realizar cálculos matemáticos",
  "parameters": {
    "type": "object",
    "properties": {
      "expressao": {
        "type": "string",
        "description": "Expressão matemática como string"
      }
    },
    "required": ["expressao"]
  }
}

FLUXO DE CHAMADA:
User: "Quanto é 10 + 5?"
  ↓
LLM vê o schema e pensa: "Tenho uma ferramenta 'calculadora' que pode fazer isso!"
  ↓
LLM retorna: AIMessage(tool_calls=[{"name": "calculadora", "args": {"expressao": "10+5"}}])
  ↓
Sistema executa: calculadora("10+5")
  ↓
Ferramenta retorna: "Resultado: 15"
  ↓
LLM vê o resultado e responde: "O resultado de 10 + 5 é 15"
"""

# ============================================================================
# 🗺️ SEÇÃO 6: DEFINIÇÃO DO ESTADO DO GRAFO
# ============================================================================

class AgentState(TypedDict):
    """
    Estado compartilhado entre todos os nós do grafo.
    
    O QUE É UM ESTADO?
    É um dicionário que passa por todos os nós do grafo, sendo
    atualizado em cada etapa. Pense nele como a "memória" do agente.
    
    FLUXO DO ESTADO:
    
    Estado inicial:
    {
      "messages": [HumanMessage("Calcule 10+5")],
      "contexto": "",
      "explicacao": "",
      "resposta_final": ""
    }
    
    ↓ Passa pelo nó "agente"
    
    {
      "messages": [
        HumanMessage("Calcule 10+5"),
        AIMessage(tool_calls=[...])  ← LLM decidiu usar tool
      ],
      "contexto": "",
      "explicacao": "",
      "resposta_final": ""
    }
    
    ↓ Passa pelo nó "ferramentas"
    
    {
      "messages": [
        HumanMessage("Calcule 10+5"),
        AIMessage(tool_calls=[...]),
        ToolMessage("Resultado: 15")  ← Tool executou
      ],
      "contexto": "",
      "explicacao": "",
      "resposta_final": ""
    }
    
    ↓ Passa pelo nó "formatar"
    
    {
      "messages": [...],
      "contexto": "",
      "explicacao": "",
      "resposta_final": "Foi solicitado o cálculo..."  ← Preenchido!
    }
    """
    
    # Lista de mensagens da conversa
    messages: Annotated[Sequence[BaseMessage], add_messages]
    """
    ENTENDENDO Annotated[Sequence[BaseMessage], add_messages]:
    
    - Sequence[BaseMessage]: Uma lista de mensagens (HumanMessage, AIMessage, etc)
    
    - Annotated[..., add_messages]: O segundo parâmetro (add_messages) é uma
      "função reducer" que define COMO combinar valores quando múltiplos nós
      atualizam o mesmo campo.
      
    EXEMPLO DE add_messages EM AÇÃO:
    
    Nó A retorna: {"messages": [AIMessage("Vou calcular")]}
    Nó B retorna: {"messages": [ToolMessage("Resultado: 15")]}
    
    add_messages combina: [AIMessage("Vou calcular"), ToolMessage("Resultado: 15")]
    
    Sem add_messages, o segundo valor substituiria o primeiro (comportamento padrão)!
    """
    
    # Campos auxiliares (não usados atualmente, mas disponíveis para expansão)
    contexto: str       # Pode armazenar contexto extraído
    explicacao: str     # Pode armazenar explicação intermediária
    resposta_final: str # Armazena a resposta formatada final

"""
EXPANDINDO O ESTADO:

Se quiser adicionar mais campos ao estado:

class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]
    contexto: str
    explicacao: str
    resposta_final: str
    
    # NOVOS CAMPOS:
    usuario_nome: str           # Nome do usuário
    sessao_id: str             # ID da sessão
    tools_usadas: list         # Lista de tools que foram usadas
    tempo_execucao: float      # Quanto tempo levou
    historico_completo: list   # Histórico de todas as interações

E use nos nós:

def meu_node(state: AgentState):
    return {
        "messages": [...],
        "usuario_nome": "João",
        "tools_usadas": ["calculadora"]
    }
"""

# ============================================================================
# 🎭 SEÇÃO 7: FUNÇÕES DOS NÓS (CÉREBRO DO GRAFO)
# ============================================================================

def agente_node(state: AgentState):
    """
    NÓ PRINCIPAL: Decide se precisa usar ferramentas ou responder.
    
    RESPONSABILIDADES:
    1. Recebe o estado atual (com histórico de mensagens)
    2. Envia para o LLM (que conhece as ferramentas disponíveis)
    3. LLM analisa e decide:
       - Opção A: Responder diretamente
       - Opção B: Usar uma ou mais ferramentas
    4. Retorna a decisão do LLM (nova mensagem no estado)
    
    EXEMPLO DE EXECUÇÃO:
    
    Input state:
    {
      "messages": [HumanMessage("Calcule 10+5")]
    }
    
    LLM analisa e decide usar a ferramenta:
    response = AIMessage(
      content="",
      tool_calls=[
        {
          "name": "calculadora",
          "args": {"expressao": "10+5"}
        }
      ]
    )
    
    Output state (merged):
    {
      "messages": [
        HumanMessage("Calcule 10+5"),
        AIMessage(tool_calls=[...])
      ]
    }
    """
    messages = state["messages"]  # Extrai histórico de mensagens
    
    # Chama o LLM com acesso às ferramentas
    response = llm_with_tools.invoke(messages)
    """
    O QUE ACONTECE NO invoke()?
    
    1. Sistema envia ao LLM:
       - Histórico de mensagens
       - Schema das ferramentas disponíveis
       - Instruções de como usar as ferramentas
    
    2. LLM processa e decide:
       - "Preciso calcular algo?" → Chama calculadora
       - "Preciso traduzir algo?" → Chama tradutor
       - "Posso responder direto?" → Responde sem ferramentas
    
    3. LLM retorna uma de duas coisas:
       A) AIMessage(content="resposta direta")
       B) AIMessage(tool_calls=[{ferramenta: calculadora, args: {...}}])
    """
    
    # Retorna novo estado (será merged com o existente)
    return {"messages": [response]}
    """
    MERGE DE ESTADO:
    
    Graças ao add_messages, quando você retorna {"messages": [response]},
    o LangGraph automaticamente ADICIONA a nova mensagem ao histórico
    existente, ao invés de substituir.
    
    Estado antes:  {"messages": [HumanMessage(...)]}
    Retorno:       {"messages": [AIMessage(...)]}
    Estado depois: {"messages": [HumanMessage(...), AIMessage(...)]}
    """


def formatar_resposta_node(state: AgentState):
    """
    NÓ DE FORMATAÇÃO: Cria a resposta final estruturada.
    
    QUANDO ESTE NÓ É CHAMADO?
    Depois que o agente terminou de usar todas as ferramentas necessárias
    e já tem todas as informações para responder.
    
    RESPONSABILIDADES:
    1. Recebe histórico completo (pergunta + ferramentas + resultados)
    2. Cria um prompt instruindo o LLM sobre como formatar
    3. LLM junta tudo e formata em 3 parágrafos
    4. Retorna resposta formatada
    
    POR QUE DUAS CHAMADAS AO LLM?
    - Primeira chamada (agente_node): DECIDIR e EXECUTAR ações
    - Segunda chamada (este nó): FORMATAR resultado
    Isso garante formato consistente!
    """
    messages = state["messages"]
    # ↑ Agora o histórico tem TUDO:
    #   - Pergunta original do usuário
    #   - Decisões do LLM (tool_calls)
    #   - Resultados das ferramentas (ToolMessages)
    #   - Possíveis iterações (se usou múltiplas ferramentas)
    
    # Cria prompt de formatação
    prompt_formatacao = f"""
Com base na conversa anterior, formate a resposta em três parágrafos seguidos, sem títulos, marcadores ou tags:

Primeiro parágrafo (contexto): Descreva o que foi entendido da pergunta e quais ferramentas foram usadas, se houver.

Segundo parágrafo (explicação): Explique o raciocínio e o passo a passo de como chegou à resposta.

Terceiro parágrafo (resposta): Entregue a resposta final de forma clara e direta.

IMPORTANTE: 
- Não use títulos como "Contexto:", "Explicação:", "Resposta:"
- Não use marcadores BEGIN/END
- Apenas escreva três parágrafos naturais separados por linha em branco
- Seja conciso mas completo
- Não use fórmulas LaTeX (\\( \\)), escreva matematicamente de forma simples
"""
    
    # Adiciona o prompt ao histórico
    messages_com_prompt = list(messages) + [HumanMessage(content=prompt_formatacao)]
    """
    POR QUE ADICIONAR AO HISTÓRICO?
    
    O LLM precisa ver TODA a conversa para formatar adequadamente:
    
    messages_com_prompt = [
      HumanMessage("Calcule 10+5"),           ← Pergunta original
      AIMessage(tool_calls=[...]),            ← LLM decidiu usar tool
      ToolMessage("Resultado: 15"),           ← Tool executou
      HumanMessage("Formate em 3 parágrafos") ← Instrução de formatação
    ]
    
    Com isso, o LLM consegue:
    1. Ver a pergunta original
    2. Ver quais tools foram usadas
    3. Ver os resultados
    4. Formatar tudo de acordo com a instrução
    """
    
    # Chama o LLM para formatar (SEM ferramentas desta vez)
    resposta_formatada = llm.invoke(messages_com_prompt)
    """
    DIFERENÇA: llm vs llm_with_tools
    
    - llm_with_tools.invoke() → LLM pode chamar ferramentas
    - llm.invoke()           → LLM só responde, sem ferramentas
    
    Aqui usamos llm.invoke() porque NÃO queremos que o LLM use
    ferramentas novamente, apenas formate a resposta final.
    """
    
    # Retorna estado atualizado
    return {
        "messages": [resposta_formatada],
        "resposta_final": resposta_formatada.content
    }
    """
    DOIS CAMPOS ATUALIZADOS:
    
    1. "messages": Adiciona a resposta formatada ao histórico
    2. "resposta_final": Armazena o texto da resposta (facilita acesso)
    
    O campo "resposta_final" é conveniente para extrair só a resposta
    sem precisar percorrer todo o histórico de mensagens.
    """


def should_continue(state: AgentState):
    """
    FUNÇÃO DE ROTEAMENTO: Decide o próximo nó do grafo.
    
    QUANDO É CHAMADA?
    Depois do nó "agente", para decidir para onde ir:
    - Se o LLM quer usar ferramentas → vai para "ferramentas"
    - Se o LLM já pode responder → vai para "formatar"
    
    RETORNOS POSSÍVEIS:
    - "continue" → Vai para o nó "ferramentas"
    - "format"   → Vai para o nó "formatar"
    
    FLUXO DE DECISÃO:
    
        [agente_node executou]
                │
                ▼
        [should_continue analisa]
                │
                ├─ LLM retornou tool_calls?
                │  │
                │  ├─ SIM → return "continue"
                │  │          │
                │  │          ▼
                │  │    [nó ferramentas]
                │  │          │
                │  │          ▼
                │  │    [volta para agente]
                │  │
                │  └─ NÃO → return "format"
                │              │
                │              ▼
                │        [nó formatar]
                │              │
                │              ▼
                │           [END]
    """
    messages = state["messages"]
    last_message = messages[-1]  # Pega a última mensagem (mais recente)
    
    # Verifica se a última mensagem tem tool_calls
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        # LLM quer usar ferramentas!
        return "continue"
    
    # LLM não quer usar ferramentas, pode formatar resposta
    return "format"

"""
EXEMPLO DE EXECUÇÃO COM MÚLTIPLAS FERRAMENTAS:

Pergunta: "Calcule 10+5 e traduza 'olá'"

ITERAÇÃO 1:
- agente_node: LLM decide usar calculadora
- should_continue: "continue" (tem tool_calls)
- ferramentas: Executa calculadora("10+5") → "Resultado: 15"
- volta para agente_node

ITERAÇÃO 2:
- agente_node: LLM vê resultado, decide usar tradutor
- should_continue: "continue" (tem tool_calls)
- ferramentas: Executa tradutor("olá") → "hello"
- volta para agente_node

ITERAÇÃO 3:
- agente_node: LLM tem tudo, não precisa mais ferramentas
- should_continue: "format" (sem tool_calls)
- formatar: Cria resposta estruturada
- END

O grafo AUTOMATICAMENTE faz múltiplas iterações até ter tudo!
"""

# ============================================================================
# 🏗️ SEÇÃO 8: CONSTRUÇÃO DO GRAFO
# ============================================================================

# Cria o grafo de estado
workflow = StateGraph(AgentState)
"""
StateGraph(AgentState):
- Cria um grafo onde o estado tem a estrutura definida em AgentState
- Cada nó pode ler e modificar este estado
- O estado é passado de nó em nó como uma "memória compartilhada"
"""

# ========== ADICIONANDO NÓS ==========

workflow.add_node("agente", agente_node)
"""
add_node(nome, função):
- nome: Identificador único

SyntaxError: incomplete input (1638038219.py, line 767)